<a href="https://colab.research.google.com/github/Daniela-Alves2004/data-mining/blob/main/Pipeline_Passenger_Sstisfaction_aula_01_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Base de dados https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction



### Setup and Data Loading
This section handles the initial setup, including importing necessary libraries and loading the dataset from the provided Kaggle source.

In [34]:
import pandas as pd
import numpy as np

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif,RFE,SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report


### Import Libraries
This cell imports all the required Python libraries for data manipulation, preprocessing, and model building.
- `pandas` for data handling.
- `numpy` for numerical operations.
- `sklearn.pipeline` for creating data processing and modeling pipelines.
- `sklearn.compose.ColumnTransformer` for applying different transformations to different columns.
- `sklearn.preprocessing.StandardScaler` for feature scaling and `sklearn.impute.SimpleImputer` for handling missing values.
- `sklearn.tree.DecisionTreeClassifier` for the classification model.
- `sklearn.metrics.classification_report` for model evaluation.

In [35]:
# gerar dois dataframes
#train
train = pd.read_csv('train.csv')
#test
test = pd.read_csv('test.csv')

### Load Data
Here, the `train.csv` and `test.csv` files are loaded into pandas DataFrames named `train` and `test`, respectively. These datasets contain the features and target variable for training and testing the machine learning model.

In [36]:
train.shape,test.shape

((103904, 25), (25976, 25))

### Check Data Dimensions
This cell displays the shape (number of rows and columns) of the `train` and `test` DataFrames. This helps confirm that the data has been loaded correctly and to understand the size of the datasets.

In [37]:
train.isna().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


### Check for Missing Values in Training Data
This code calculates and displays the count of missing values for each column in the `train` DataFrame. It's an important step in exploratory data analysis to identify columns that require imputation or other missing value handling strategies.

In [38]:
test.isna().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


### Check for Missing Values in Test Data
Similar to the training data, this cell checks for and displays the count of missing values for each column in the `test` DataFrame. This ensures consistency in handling missing values across both datasets.

In [39]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

### Display Data Information
This cell provides a concise summary of the `train` DataFrame, including the data types of each column, the number of non-null values, and memory usage. This helps in understanding the data structure and identifying potential issues like incorrect data types or columns with many missing values.

In [40]:
train["satisfaction"].unique()

array(['neutral or dissatisfied', 'satisfied'], dtype=object)

### Explore Target Variable
This cell shows the unique values present in the `satisfaction` column of the `train` DataFrame. This is crucial for understanding the nature of the target variable (e.g., binary, categorical) and ensures that the label encoding is applied correctly.

In [41]:
#lIMPEZA INICIAL
#rRemocao colunas irrelevantes

remove_cols =["Unnamed: 0","id"]
train = train.drop(columns=remove_cols)
test = test.drop(columns=remove_cols)

### Initial Data Cleaning: Remove Irrelevant Columns
This step involves removing columns that are not useful for the machine learning model, such as `Unnamed: 0` (often an old index column) and `id`. These columns typically do not contribute to the predictive power of the model.

In [42]:
train.shape, test.shape

((103904, 23), (25976, 23))

### Verify Data Dimensions After Column Removal
After removing irrelevant columns, this cell re-checks and displays the shape of the `train` and `test` DataFrames to confirm that the columns were dropped as intended and the datasets maintain their integrity.

In [43]:
# gerar x_traine x_test
x_train = train.drop(columns="satisfaction")
y_train = train["satisfaction"]
x_test = test.drop(columns="satisfaction")
y_test = test["satisfaction"]

### Split Data into Features (X) and Target (y)
This cell separates the features (independent variables) from the target variable (`satisfaction`) for both the training and test datasets. `x_train` and `x_test` will contain the input features, while `y_train` and `y_test` will hold the corresponding target labels.

In [44]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((103904, 22), (103904,), (25976, 22), (25976,))

### Verify Dimensions of X and Y
This cell displays the shapes of `x_train`, `y_train`, `x_test`, and `y_test` to confirm that the splitting operation was performed correctly and that the dimensions are consistent for further processing.

In [45]:
#transformar os dados de satisfaction
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

### Encode Target Variable
This step uses `LabelEncoder` from `sklearn.preprocessing` to convert the categorical `satisfaction` target variable (e.g., 'neutral or dissatisfied', 'satisfied') into numerical labels (0 and 1). This is a necessary step as machine learning models typically require numerical input for the target variable.

In [46]:
y_train

array([0, 0, 1, ..., 0, 0, 0])

### Display Encoded Training Target
This cell displays the first few elements of the `y_train` array after label encoding, confirming that the categorical labels have been successfully converted to numerical values.

In [47]:
#separacao das colunas numericas e categoriacas
numeric_cols = x_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = x_train.select_dtypes("object").columns.tolist()

### Separate Numeric and Categorical Columns
This cell identifies and separates the column names into two lists: `numeric_cols` for numerical features and `categorical_cols` for categorical (object type) features. This separation is essential for applying different preprocessing steps (e.g., scaling for numeric, one-hot encoding for categorical).

In [48]:
categorical_cols

['Gender', 'Customer Type', 'Type of Travel', 'Class']

### Display Categorical Column Names
This cell outputs the list of identified categorical column names, allowing for verification of the column separation process.

Construcao do pipeline

### Build Preprocessing Pipeline
This section defines the preprocessing steps for both numerical and categorical features using `sklearn` pipelines.

In [49]:
from sklearn.preprocessing import OneHotEncoder
#tratar colunas numericas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
#Tratar colunas categoricas
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numumeric', numeric_transformer, numeric_cols),
        ('categoric', categorical_transformer, categorical_cols)
    ])

### Define Preprocessing Transformers
This cell constructs the preprocessing pipelines for numerical and categorical features:
- **`numeric_transformer`**: Applies `SimpleImputer` (median strategy) to handle missing numerical values, followed by `StandardScaler` to normalize numerical features.
- **`categorical_transformer`**: Uses `OneHotEncoder` to convert categorical features into a one-hot encoded numerical representation, handling unknown categories gracefully with `handle_unknown='ignore'`.
- **`preprocessor`**: A `ColumnTransformer` is used to apply these distinct transformers to their respective column types (`numeric_cols` and `categorical_cols`).

In [50]:
preprocessor

ColumnTransformer(transformers=[('numumeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Flight Distance',
                                  'Inflight wifi service',
                                  'Departure/Arrival time convenient',
                                  'Ease of Online booking', 'Gate location',
                                  'Food and drink', 'Online boarding',
                                  'Seat comfort', 'Inflight entertainment',
                                  'On-board service', 'Leg room service',
                                  'Baggage handling', 'Checkin service',
                                  'Inflight service', 'Cleanliness',
                                  'Departure Delay in Minutes',
                                  'Arrival Delay in Minutes']),
                                ('categoric',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Customer Type', 'Type of Travel',
                                  'Class'])])

### Display Preprocessor Configuration
This cell displays the configured `ColumnTransformer` (`preprocessor`), providing a summary of which transformations will be applied to which columns. This is useful for reviewing the preprocessing setup.

In [51]:
#treinamento
pipelineExec = Pipeline(steps=
                        [("prepoc",preprocessor),
                        ("clf",DecisionTreeClassifier(random_state=42))
                        ])

### Build Machine Learning Pipeline
This cell constructs the full machine learning pipeline, combining the preprocessing steps with the classification model:
- The `preprocessor` (defined earlier) handles data transformation.
- A `DecisionTreeClassifier` is chosen as the classifier, with `random_state=42` for reproducibility.

In [52]:
pipelineExec

Pipeline(steps=[('prepoc',
                 ColumnTransformer(transformers=[('numumeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Flight Distance',
                                                   'Inflight wifi service',
                                                   'Departure/Arrival time '
                                                   'convenient',
                                                   'Ease of Online booking',
                                                   'Gate location',
                                                   'Food and drink',
                                                   'Online boarding',
                                                   'Seat comfort',
                                                   'Inflight entertainment',
                                                   'On-board service',
                                                   'Leg room service',
                                                   'Baggage handling',
                                                   'Checkin service',
                                                   'Inflight service',
                                                   'Cleanliness',
                                                   'Departure Delay in Minutes',
                                                   'Arrival Delay in Minutes']),
                                                 ('categoric',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Customer Type',
                                                   'Type of Travel',
                                                   'Class'])])),
                ('clf', DecisionTreeClassifier(random_state=42))])

### Display Full Pipeline Configuration
This cell outputs the complete `pipelineExec` object, showing all the sequential steps defined, from preprocessing to the final classifier. This helps in understanding the entire workflow for model training and prediction.

In [53]:
#trienamento
pipelineExec.fit(x_train,y_train)

Pipeline(steps=[('prepoc',
                 ColumnTransformer(transformers=[('numumeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Flight Distance',
                                                   'Inflight wifi service',
                                                   'Departure/Arrival time '
                                                   'convenient',
                                                   'Ease of Online booking',
                                                   'Gate location',
                                                   'Food and drink',
                                                   'Online boarding',
                                                   'Seat comfort',
                                                   'Inflight entertainment',
                                                   'On-board service',
                                                   'Leg room service',
                                                   'Baggage handling',
                                                   'Checkin service',
                                                   'Inflight service',
                                                   'Cleanliness',
                                                   'Departure Delay in Minutes',
                                                   'Arrival Delay in Minutes']),
                                                 ('categoric',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Customer Type',
                                                   'Type of Travel',
                                                   'Class'])])),
                ('clf', DecisionTreeClassifier(random_state=42))])

### Train the Model
This cell initiates the training process by calling the `fit` method on the `pipelineExec`. The pipeline will first apply the defined preprocessing steps to `x_train` and then train the `DecisionTreeClassifier` using the processed features and `y_train` labels.

In [54]:
y_pred = pipelineExec.predict(x_test)

### Make Predictions
After training, this cell uses the fitted `pipelineExec` to make predictions on the unseen `x_test` dataset. The pipeline automatically applies the same preprocessing steps to `x_test` before feeding it to the trained `DecisionTreeClassifier`.

In [55]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95     14573
           1       0.94      0.94      0.94     11403

    accuracy                           0.95     25976
   macro avg       0.95      0.95      0.95     25976
weighted avg       0.95      0.95      0.95     25976



### Evaluate Model Performance
This cell generates and prints a `classification_report`, which provides a detailed summary of the model's performance on the test set. It includes precision, recall, f1-score, and support for each class, as well as overall accuracy, macro average, and weighted average.